# Llama 3.1 70B on Amazon Bedrock

Llama 3.1 70B Instruct is an open-weights model from Meta. In this lab you'll call it through **Amazon Bedrock**, a fully managed service, so no GPU is needed: your session only sends requests.

You'll learn how to:

* Send a request with plain HTTP (`curl`)
* Call the model from Python with `boto3` and the Bedrock **Converse API**
* Stream tokens as they are generated
* Keep a multi-turn conversation
* Build an end-to-end chat UI with Gradio

**Before you start**, make sure these environment variables are set in your project (Project Settings > Advanced > Environment Variables) and that you started a **new session** after setting them:

| Variable | Example |
|---|---|
| `AWS_BEARER_TOKEN_BEDROCK` | Your Bedrock API key |
| `AWS_REGION` | `us-east-1` |
| `BEDROCK_MODEL_ID` | `us.meta.llama3-1-70b-instruct-v1:0` |

In [ ]:
import os

for var in ["AWS_BEARER_TOKEN_BEDROCK", "AWS_REGION", "BEDROCK_MODEL_ID"]:
    print(var, "is set" if os.environ.get(var) else "is MISSING")

## Doing curl requests

The Converse API takes a list of **messages**, each with a `role` (`user` or `assistant`) and a list of `content` blocks. You don't need special prompt tokens such as `[INST]`; Bedrock applies the model's chat template for you.

With a Bedrock API key, you authenticate with a standard `Authorization: Bearer` header.

In [ ]:
%%bash
curl -s "https://bedrock-runtime.${AWS_REGION}.amazonaws.com/model/${BEDROCK_MODEL_ID:-us.meta.llama3-1-70b-instruct-v1:0}/converse" \
  -H "Content-Type: application/json" \
  -H "Authorization: Bearer $AWS_BEARER_TOKEN_BEDROCK" \
  -d '{"messages": [{"role": "user", "content": [{"text": "Explain ML as a pirate in two sentences"}]}], "inferenceConfig": {"maxTokens": 100}}'

## Programmatic usage with Python

`boto3`, the AWS SDK for Python, picks up the API key from the `AWS_BEARER_TOKEN_BEDROCK` environment variable automatically (boto3 1.39.0 or later). The **Converse API** works the same way for every chat model on Bedrock, and lets you set generation parameters such as `temperature`, `topP` and `maxTokens`.

In [ ]:
%%capture
!pip install "boto3>=1.39.0" "gradio==4.44.1"

In [ ]:
import os
import boto3

region = os.environ.get("AWS_REGION", "us-east-1")
model_id = os.environ.get("BEDROCK_MODEL_ID", "us.meta.llama3-1-70b-instruct-v1:0")

client = boto3.client("bedrock-runtime", region_name=region)

res = client.converse(
    modelId=model_id,
    messages=[{"role": "user", "content": [{"text": "What is your favourite condiment?"}]}],
    inferenceConfig={"maxTokens": 95},
)
print(res["output"]["message"]["content"][0]["text"])
print(res["usage"])

We can also use **token streaming**: the server returns tokens as they are generated. Use `converse_stream` and loop over the events.

In [ ]:
response = client.converse_stream(
    modelId=model_id,
    messages=[{"role": "user", "content": [{"text": "What is your favourite condiment?"}]}],
    inferenceConfig={"maxTokens": 35},
)

for event in response["stream"]:
    # print each event to see what the stream contains
    print(event)

Let's now try a multi-turn conversation. The model has no memory between calls, so we send the whole history every time as a list of messages.

In [ ]:
def format_prompt(message, history):
    messages = []
    for user_prompt, bot_response in history:
        messages.append({"role": "user", "content": [{"text": user_prompt}]})
        messages.append({"role": "assistant", "content": [{"text": bot_response}]})
    messages.append({"role": "user", "content": [{"text": message}]})
    return messages

In [ ]:
message = "And what do you think about it?"
history = [["What is your favourite condiment?", "My favorite condiment is ketchup. It's versatile, tasty, and goes well with a variety of foods."]]

format_prompt(message, history)

## End-to-end demo

Let's now build a Gradio demo that takes care of:

* Handling multiple turns of conversation
* Formatting the messages in the correct structure
* Letting the user change the generation parameters
* Streaming the reply

Just run the following cell and have fun!

In [ ]:
import gradio as gr

def generate(prompt, history, temperature=0.9, max_new_tokens=256, top_p=0.95):
    temperature = max(float(temperature), 1e-2)
    response = client.converse_stream(
        modelId=model_id,
        messages=format_prompt(prompt, history),
        inferenceConfig={"temperature": temperature, "topP": float(top_p), "maxTokens": int(max_new_tokens)},
    )
    output = ""
    for event in response["stream"]:
        delta = event.get("contentBlockDelta", {}).get("delta", {})
        if "text" in delta:
            output += delta["text"]
            yield output
    return output


additional_inputs = [
    gr.Slider(label="Temperature", value=0.9, minimum=0.0, maximum=1.0, step=0.05, interactive=True,
              info="Higher values produce more diverse outputs"),
    gr.Slider(label="Max new tokens", value=256, minimum=64, maximum=1048, step=64, interactive=True,
              info="The maximum numbers of new tokens"),
    gr.Slider(label="Top-p (nucleus sampling)", value=0.90, minimum=0.0, maximum=1.0, step=0.05, interactive=True,
              info="Higher values sample more low-probability tokens"),
]

with gr.Blocks() as demo:
    gr.ChatInterface(generate, additional_inputs=additional_inputs)

demo.queue().launch(debug=True)

## What's next?

* Run the full app as a CML Application (`app.py`) to share the chatbot with others
* Try a bigger model: set `BEDROCK_MODEL_ID` to another Bedrock model ID (for example a smaller Llama model, shown in the Bedrock model catalog) and compare the answers
* Read the [Llama 3.1 70B model card on Bedrock](https://docs.aws.amazon.com/bedrock/latest/userguide/models-supported.html)